# 10 - Report Language and Similarity Analysis

This notebook provides an exploratory linguistic and split-leakage audit for
the complete 2,300-report RadGraph-XL collection. It measures lexical style,
negation and uncertainty cues, annotation density, exact normalised-text
duplicates and TF-IDF cosine similarity.

Restricted report text is processed only in memory. Outputs contain aggregate
terms, report identifiers, hashes, numeric metrics and figures; no report text
or token sequence is written or displayed. Similarity is evidence of lexical
overlap or templating, not proof that two reports have the same clinical meaning.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import FeatureUnion


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

MIMIC_ZIP = Path(os.environ["RADGRAPH_XL_MIMIC_ZIP"])
STANFORD_JSONL = Path(os.environ["RADGRAPH_XL_STANFORD_JSONL"])
RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
if RUN_NAME != "full_2300":
    raise ValueError(
        "Notebook 10 is the complete-dataset language audit and requires "
        "RADGRAPH_XL_RUN_NAME=full_2300"
    )

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
AUDIT_DIR = OUTPUT_ROOT / "audit"
INTERIM_DIR = OUTPUT_ROOT / "interim"
LINGUISTIC_DIR = OUTPUT_ROOT / "linguistic_audit"
FIGURE_DIR = OUTPUT_ROOT / "figures"
LINGUISTIC_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = AUDIT_DIR / "data_audit.json"
SPLIT_PATH = INTERIM_DIR / "report_splits.csv"
assert MIMIC_ZIP.exists(), MIMIC_ZIP
assert STANFORD_JSONL.exists(), STANFORD_JSONL
assert AUDIT_PATH.exists(), "Run Notebook 01 first"
assert SPLIT_PATH.exists(), "Run Notebook 03 first to create the fixed report split"

EXPECTED_REPORTS = 2300
SIMILARITY_THRESHOLDS = [0.80, 0.90, 0.95, 0.98]
NEGATION_CUES = {
    "absent", "absence", "cannot", "denied", "denies", "negative",
    "neither", "no", "nor", "not", "without",
}
UNCERTAINTY_CUES = {
    "could", "likely", "may", "might", "possible", "possibly",
    "probably", "questionable", "suggestive", "suspected",
}
CLINICAL_STOP_WORDS = sorted(
    set(ENGLISH_STOP_WORDS) - {"no", "not", "without", "may", "could"}
)

sns.set_theme(style="whitegrid", context="notebook")
print(
    {
        "run_name": RUN_NAME,
        "dataset_scope": "complete 2,300-report RadGraph-XL collection",
        "linguistic_output": str(LINGUISTIC_DIR),
        "figure_output": str(FIGURE_DIR),
    }
)


In [ ]:
def read_jsonl_lines(lines, source: str) -> list[dict]:
    records = []
    for raw_line in lines:
        if not raw_line.strip():
            continue
        record = json.loads(raw_line)
        dataset = str(record["dataset"])
        records.append(
            {
                "source": source,
                "dataset": dataset,
                "doc_id": f"{dataset}::{record['doc_key']}",
                "record": record,
            }
        )
    return records


with zipfile.ZipFile(MIMIC_ZIP) as archive:
    members = [name for name in archive.namelist() if name.lower().endswith(".jsonl")]
    if len(members) != 1:
        raise ValueError(f"Expected one MIMIC JSONL member, found {members}")
    with archive.open(members[0]) as handle:
        mimic_records = read_jsonl_lines(handle, "mimic")

with STANFORD_JSONL.open("rb") as handle:
    stanford_records = read_jsonl_lines(handle, "stanford")

loaded_records = mimic_records + stanford_records
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
if len(loaded_records) != EXPECTED_REPORTS or audit.get("reports") != EXPECTED_REPORTS:
    raise ValueError(
        f"Expected {EXPECTED_REPORTS} reports; raw={len(loaded_records)}, "
        f"audit={audit.get('reports')}"
    )

split_df = pd.read_csv(SPLIT_PATH)
if len(split_df) != EXPECTED_REPORTS or split_df["doc_id"].nunique() != EXPECTED_REPORTS:
    raise ValueError("The fixed report split must contain 2,300 unique report identifiers")
split_map = split_df.set_index("doc_id")["split"].to_dict()

raw_ids = {item["doc_id"] for item in loaded_records}
if raw_ids != set(split_map):
    raise ValueError("Raw report identifiers and fixed-split report identifiers do not match")

print(
    {
        "mimic": len(mimic_records),
        "stanford": len(stanford_records),
        "combined": len(loaded_records),
        "split_counts": split_df["split"].value_counts().to_dict(),
    }
)


In [ ]:
def normalise_lexical_token(token: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", token.casefold())


def normalise_for_hash(tokens: list[str]) -> str:
    text = " ".join(token.casefold() for token in tokens)
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def moving_average_type_token_ratio(tokens: list[str], window: int = 50) -> float:
    lexical = [normalise_lexical_token(token) for token in tokens]
    lexical = [token for token in lexical if token]
    if not lexical:
        return 0.0
    if len(lexical) <= window:
        return len(set(lexical)) / len(lexical)

    counts = Counter(lexical[:window])
    scores = [len(counts) / window]
    for index in range(window, len(lexical)):
        outgoing = lexical[index - window]
        counts[outgoing] -= 1
        if counts[outgoing] == 0:
            del counts[outgoing]
        counts[lexical[index]] += 1
        scores.append(len(counts) / window)
    return float(np.mean(scores))


report_rows = []
documents = []
normalised_hash_texts = []

for item in loaded_records:
    record = item["record"]
    tokens = [str(token) for sentence in record["sentences"] for token in sentence]
    lexical_tokens = [normalise_lexical_token(token) for token in tokens]
    lexical_tokens = [token for token in lexical_tokens if token]
    entities = [annotation for sentence in record["ner"] for annotation in sentence]
    relations = [annotation for sentence in record["relations"] for annotation in sentence]
    token_count = len(tokens)
    normalised_text = normalise_for_hash(tokens)

    report_rows.append(
        {
            "doc_id": item["doc_id"],
            "source": item["source"],
            "dataset": item["dataset"],
            "split": split_map[item["doc_id"]],
            "token_count": token_count,
            "unique_lexical_tokens": len(set(lexical_tokens)),
            "mattr_50": moving_average_type_token_ratio(tokens, window=50),
            "mean_lexical_token_length": float(np.mean([len(token) for token in lexical_tokens])),
            "uppercase_tokens_per_100": 100 * sum(
                token.isupper() and len(token) > 1 for token in tokens
            ) / token_count,
            "negation_cues_per_100": 100 * sum(
                token in NEGATION_CUES for token in lexical_tokens
            ) / token_count,
            "uncertainty_cues_per_100": 100 * sum(
                token in UNCERTAINTY_CUES for token in lexical_tokens
            ) / token_count,
            "entities_per_100_tokens": 100 * len(entities) / token_count,
            "relations_per_100_tokens": 100 * len(relations) / token_count,
            "normalised_text_sha256": hashlib.sha256(
                normalised_text.encode("utf-8")
            ).hexdigest(),
        }
    )
    documents.append(" ".join(tokens))
    normalised_hash_texts.append(normalised_text)

language_df = pd.DataFrame(report_rows)
language_metrics_path = LINGUISTIC_DIR / "language_metrics_by_report.csv"
language_df.to_csv(language_metrics_path, index=False)

print(
    language_df.groupby("dataset").agg(
        reports=("doc_id", "size"),
        median_tokens=("token_count", "median"),
        mean_mattr_50=("mattr_50", "mean"),
        mean_negation_per_100=("negation_cues_per_100", "mean"),
        mean_uncertainty_per_100=("uncertainty_cues_per_100", "mean"),
    ).round(3)
)


In [ ]:
metric_columns = [
    "token_count",
    "mattr_50",
    "mean_lexical_token_length",
    "uppercase_tokens_per_100",
    "negation_cues_per_100",
    "uncertainty_cues_per_100",
    "entities_per_100_tokens",
    "relations_per_100_tokens",
]
group_metrics = language_df.groupby(["source", "dataset"])[metric_columns].agg(
    ["mean", "median", "std"]
)
group_metrics.columns = [f"{metric}_{stat}" for metric, stat in group_metrics.columns]
group_metrics = group_metrics.reset_index()
group_metrics_path = LINGUISTIC_DIR / "language_metrics_by_dataset.csv"
group_metrics.to_csv(group_metrics_path, index=False)

top_term_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=CLINICAL_STOP_WORDS,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    max_features=30000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z-]+\b",
    sublinear_tf=True,
)
top_term_matrix = top_term_vectorizer.fit_transform(documents)
feature_names = np.asarray(top_term_vectorizer.get_feature_names_out())
top_term_rows = []
for dataset in sorted(language_df["dataset"].unique()):
    mask = language_df["dataset"].eq(dataset).to_numpy()
    scores = np.asarray(top_term_matrix[mask].mean(axis=0)).ravel()
    best_indices = scores.argsort()[-12:][::-1]
    for rank, feature_index in enumerate(best_indices, start=1):
        top_term_rows.append(
            {
                "dataset": dataset,
                "rank": rank,
                "term": feature_names[feature_index],
                "mean_tfidf": float(scores[feature_index]),
            }
        )

top_terms_df = pd.DataFrame(top_term_rows)
top_terms_path = LINGUISTIC_DIR / "top_terms_by_dataset.csv"
top_terms_df.to_csv(top_terms_path, index=False)

print("Wrote aggregate language tables:")
print(" -", language_metrics_path)
print(" -", group_metrics_path)
print(" -", top_terms_path)


In [ ]:
display_metrics = {
    "token_count": "Report length",
    "mattr_50": "Lexical diversity (MATTR-50)",
    "negation_cues_per_100": "Negation cues / 100 tokens",
    "uncertainty_cues_per_100": "Uncertainty cues / 100 tokens",
    "entities_per_100_tokens": "Entities / 100 tokens",
    "relations_per_100_tokens": "Relations / 100 tokens",
}
language_means = language_df.groupby("dataset")[list(display_metrics)].mean()
standardised = (language_means - language_means.mean()) / language_means.std(ddof=0)
standardised = standardised.rename(columns=display_metrics)

plt.figure(figsize=(11, 5.5))
sns.heatmap(standardised, cmap="vlag", center=0, linewidths=0.5, cbar_kws={"label": "Standard deviations from group mean"})
plt.title("Standardised Report-Language Characteristics by Source-Modality Group")
plt.xlabel("Language or annotation characteristic")
plt.ylabel("Dataset group")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
language_figure_path = FIGURE_DIR / "report_language_characteristics_by_dataset.png"
plt.savefig(language_figure_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close()
print(language_figure_path)


In [ ]:
datasets = sorted(top_terms_df["dataset"].unique())
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for axis, dataset in zip(axes.flat, datasets):
    subset = top_terms_df[top_terms_df["dataset"] == dataset].sort_values("mean_tfidf")
    axis.barh(subset["term"], subset["mean_tfidf"], color="#0B6E69")
    axis.set_title(dataset)
    axis.set_xlabel("Mean TF-IDF")
    axis.set_ylabel("")
for axis in axes.flat[len(datasets):]:
    axis.axis("off")
fig.suptitle("Highest-Weighted Lexical Terms by Source-Modality Group", fontsize=15, y=1.01)
plt.tight_layout()
top_terms_figure_path = FIGURE_DIR / "top_terms_by_dataset.png"
plt.savefig(top_terms_figure_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close()
print(top_terms_figure_path)


In [ ]:
duplicate_rows = []
duplicate_group_count = 0
cross_split_exact_group_count = 0
for text_hash, group in language_df.groupby("normalised_text_sha256"):
    if len(group) < 2:
        continue
    duplicate_group_count += 1
    crosses_split = group["split"].nunique() > 1
    cross_split_exact_group_count += int(crosses_split)
    group_id = f"exact-{duplicate_group_count:04d}"
    for row in group.itertuples(index=False):
        duplicate_rows.append(
            {
                "duplicate_group": group_id,
                "normalised_text_sha256": text_hash,
                "doc_id": row.doc_id,
                "source": row.source,
                "dataset": row.dataset,
                "split": row.split,
                "cross_split_group": crosses_split,
            }
        )

duplicate_columns = [
    "duplicate_group", "normalised_text_sha256", "doc_id", "source",
    "dataset", "split", "cross_split_group",
]
exact_duplicates_df = pd.DataFrame(duplicate_rows, columns=duplicate_columns)
exact_duplicates_path = LINGUISTIC_DIR / "exact_normalised_duplicate_groups.csv"
exact_duplicates_df.to_csv(exact_duplicates_path, index=False)

print(
    {
        "exact_normalised_duplicate_groups": duplicate_group_count,
        "reports_in_exact_duplicate_groups": len(exact_duplicates_df),
        "cross_split_exact_duplicate_groups": cross_split_exact_group_count,
        "output": str(exact_duplicates_path),
    }
)


In [ ]:
similarity_vectorizer = FeatureUnion(
    [
        (
            "word",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.995,
                max_features=40000,
                sublinear_tf=True,
            ),
        ),
        (
            "character",
            TfidfVectorizer(
                lowercase=True,
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=3,
                max_features=30000,
                sublinear_tf=True,
            ),
        ),
    ]
)
similarity_features = similarity_vectorizer.fit_transform(documents)
similarity_matrix = cosine_similarity(similarity_features).astype(np.float32)
np.fill_diagonal(similarity_matrix, -1.0)

split_values = language_df["split"].to_numpy()
best_any_index = similarity_matrix.argmax(axis=1)
best_any_similarity = similarity_matrix[np.arange(EXPECTED_REPORTS), best_any_index]

cross_split_matrix = similarity_matrix.copy()
same_split_mask = split_values[:, None] == split_values[None, :]
cross_split_matrix[same_split_mask] = -1.0
best_cross_index = cross_split_matrix.argmax(axis=1)
best_cross_similarity = cross_split_matrix[np.arange(EXPECTED_REPORTS), best_cross_index]

nearest_df = language_df[["doc_id", "source", "dataset", "split"]].copy()
nearest_df["nearest_doc_id"] = language_df.iloc[best_any_index]["doc_id"].to_numpy()
nearest_df["nearest_similarity"] = best_any_similarity
nearest_df["nearest_cross_split_doc_id"] = language_df.iloc[best_cross_index]["doc_id"].to_numpy()
nearest_df["nearest_cross_split_split"] = split_values[best_cross_index]
nearest_df["nearest_cross_split_similarity"] = best_cross_similarity
nearest_path = LINGUISTIC_DIR / "nearest_report_similarity.csv"
nearest_df.to_csv(nearest_path, index=False)

print(
    {
        "feature_matrix_shape": similarity_features.shape,
        "median_max_cross_split_similarity": float(np.median(best_cross_similarity)),
        "p95_max_cross_split_similarity": float(np.quantile(best_cross_similarity, 0.95)),
        "maximum_cross_split_similarity": float(best_cross_similarity.max()),
        "output": str(nearest_path),
    }
)


In [ ]:
upper_i, upper_j = np.triu_indices(EXPECTED_REPORTS, k=1)
pair_similarity = similarity_matrix[upper_i, upper_j]
cross_split_pairs = split_values[upper_i] != split_values[upper_j]

threshold_rows = []
for threshold in SIMILARITY_THRESHOLDS:
    at_threshold = pair_similarity >= threshold
    threshold_rows.append(
        {
            "threshold": threshold,
            "all_report_pairs": int(at_threshold.sum()),
            "cross_split_report_pairs": int((at_threshold & cross_split_pairs).sum()),
        }
    )

threshold_df = pd.DataFrame(threshold_rows)
threshold_path = LINGUISTIC_DIR / "similarity_threshold_summary.csv"
threshold_df.to_csv(threshold_path, index=False)

near_mask = pair_similarity >= 0.95
near_i = upper_i[near_mask]
near_j = upper_j[near_mask]
near_pairs_df = pd.DataFrame(
    {
        "doc_id_a": language_df.iloc[near_i]["doc_id"].to_numpy(),
        "source_a": language_df.iloc[near_i]["source"].to_numpy(),
        "dataset_a": language_df.iloc[near_i]["dataset"].to_numpy(),
        "split_a": split_values[near_i],
        "doc_id_b": language_df.iloc[near_j]["doc_id"].to_numpy(),
        "source_b": language_df.iloc[near_j]["source"].to_numpy(),
        "dataset_b": language_df.iloc[near_j]["dataset"].to_numpy(),
        "split_b": split_values[near_j],
        "cosine_similarity": pair_similarity[near_mask],
        "cross_split": cross_split_pairs[near_mask],
        "exact_normalised_text": (
            language_df.iloc[near_i]["normalised_text_sha256"].to_numpy()
            == language_df.iloc[near_j]["normalised_text_sha256"].to_numpy()
        ),
    }
).sort_values("cosine_similarity", ascending=False)
near_pairs_path = LINGUISTIC_DIR / "near_duplicate_pairs_ge_095.csv"
near_pairs_df.to_csv(near_pairs_path, index=False)

print(threshold_df.to_string(index=False))
print("Near-duplicate audit:", near_pairs_path)


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(best_cross_similarity, bins=45, color="#4C72B0")
for threshold, colour in [(0.90, "#DD8452"), (0.95, "#C44E52")]:
    plt.axvline(threshold, color=colour, linestyle="--", linewidth=1.5, label=f"{threshold:.2f}")
plt.title("Maximum Cross-Split TF-IDF Cosine Similarity per Report")
plt.xlabel("Maximum similarity to a report in another split")
plt.ylabel("Reports")
plt.legend(title="Audit threshold")
plt.tight_layout()
cross_split_figure_path = FIGURE_DIR / "max_cross_split_report_similarity.png"
plt.savefig(cross_split_figure_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close()
print(cross_split_figure_path)


In [ ]:
dataset_names = sorted(language_df["dataset"].unique())
dataset_similarity = pd.DataFrame(index=dataset_names, columns=dataset_names, dtype=float)
for dataset_a in dataset_names:
    index_a = np.flatnonzero(language_df["dataset"].eq(dataset_a).to_numpy())
    for dataset_b in dataset_names:
        index_b = np.flatnonzero(language_df["dataset"].eq(dataset_b).to_numpy())
        block = similarity_matrix[np.ix_(index_a, index_b)]
        if dataset_a == dataset_b:
            values = block[~np.eye(len(index_a), dtype=bool)]
        else:
            values = block.ravel()
        dataset_similarity.loc[dataset_a, dataset_b] = float(values.mean())

dataset_similarity_path = LINGUISTIC_DIR / "mean_similarity_by_dataset.csv"
dataset_similarity.to_csv(dataset_similarity_path)

plt.figure(figsize=(9, 7))
sns.heatmap(dataset_similarity, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=float(dataset_similarity.to_numpy().max()))
plt.title("Mean TF-IDF Cosine Similarity by Source-Modality Group")
plt.xlabel("Dataset group")
plt.ylabel("Dataset group")
plt.tight_layout()
dataset_similarity_figure_path = FIGURE_DIR / "mean_report_similarity_by_dataset.png"
plt.savefig(dataset_similarity_figure_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close()
print(dataset_similarity_figure_path)


In [ ]:
threshold_long = threshold_df.melt(
    id_vars="threshold",
    value_vars=["all_report_pairs", "cross_split_report_pairs"],
    var_name="pair_scope",
    value_name="pairs",
)
threshold_long["pair_scope"] = threshold_long["pair_scope"].map(
    {
        "all_report_pairs": "All report pairs",
        "cross_split_report_pairs": "Cross-split pairs",
    }
)

plt.figure(figsize=(8, 4.8))
ax = sns.barplot(data=threshold_long, x="threshold", y="pairs", hue="pair_scope", palette=["#4C72B0", "#C44E52"])
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.title("Report Pairs above TF-IDF Similarity Thresholds")
plt.xlabel("Cosine similarity threshold")
plt.ylabel("Report pairs")
plt.ylim(0, max(10, int(threshold_long["pairs"].max()) + 2))
plt.legend(title="Pair scope")
plt.tight_layout()
threshold_figure_path = FIGURE_DIR / "report_similarity_threshold_counts.png"
plt.savefig(threshold_figure_path, dpi=220, bbox_inches="tight")
plt.show()
plt.close()
print(threshold_figure_path)


In [ ]:
summary = {
    "generated_by": "notebooks/10_report_language_and_similarity_analysis.ipynb",
    "run_name": RUN_NAME,
    "dataset_scope": "complete 2,300-report RadGraph-XL collection",
    "reports": EXPECTED_REPORTS,
    "privacy": {
        "raw_report_text_exported": False,
        "token_sequences_exported": False,
        "aggregate_terms_exported": True,
        "report_ids_hashes_and_numeric_scores_exported": True,
    },
    "language_metrics": {
        "mattr_window_tokens": 50,
        "negation_cues": sorted(NEGATION_CUES),
        "uncertainty_cues": sorted(UNCERTAINTY_CUES),
    },
    "similarity_method": {
        "representation": "equal-weight union of word (1-2 gram) and character (3-5 gram) TF-IDF",
        "metric": "cosine similarity",
        "thresholds": SIMILARITY_THRESHOLDS,
        "interpretation": "lexical overlap or templating; not clinical-semantic equivalence",
    },
    "exact_normalised_duplicate_groups": duplicate_group_count,
    "reports_in_exact_duplicate_groups": len(exact_duplicates_df),
    "cross_split_exact_duplicate_groups": cross_split_exact_group_count,
    "similarity_threshold_counts": threshold_rows,
    "maximum_cross_split_similarity_summary": {
        "median": float(np.median(best_cross_similarity)),
        "p95": float(np.quantile(best_cross_similarity, 0.95)),
        "maximum": float(best_cross_similarity.max()),
    },
    "outputs": {
        "language_metrics_by_report": str(language_metrics_path),
        "language_metrics_by_dataset": str(group_metrics_path),
        "top_terms_by_dataset": str(top_terms_path),
        "exact_duplicate_groups": str(exact_duplicates_path),
        "nearest_report_similarity": str(nearest_path),
        "similarity_threshold_summary": str(threshold_path),
        "near_duplicate_pairs_ge_095": str(near_pairs_path),
        "mean_similarity_by_dataset": str(dataset_similarity_path),
    },
    "figures": {
        "report_language_characteristics_by_dataset.png": {
            "path": str(language_figure_path),
            "recommended_placement": "Methodology: Report Language and Similarity Audit",
            "caption": "Standardised lexical and annotation characteristics across the seven source-modality groups.",
        },
        "max_cross_split_report_similarity.png": {
            "path": str(cross_split_figure_path),
            "recommended_placement": "Methodology or Results validity subsection",
            "caption": "Distribution of each report's maximum TF-IDF cosine similarity to a report in another fixed partition.",
        },
        "top_terms_by_dataset.png": {
            "path": str(top_terms_figure_path),
            "recommended_placement": "Appendix",
            "caption": "Highest-weighted aggregate lexical terms by source-modality group.",
        },
        "mean_report_similarity_by_dataset.png": {
            "path": str(dataset_similarity_figure_path),
            "recommended_placement": "Appendix",
            "caption": "Mean TF-IDF cosine similarity within and between source-modality groups.",
        },
        "report_similarity_threshold_counts.png": {
            "path": str(threshold_figure_path),
            "recommended_placement": "Appendix",
            "caption": "Counts of all and cross-split report pairs above prespecified similarity thresholds.",
        },
    },
}
summary_path = LINGUISTIC_DIR / "language_similarity_audit.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Language and similarity audit complete:")
print(" - summary:", summary_path)
print(" - exact duplicate groups:", duplicate_group_count)
print(" - cross-split exact duplicate groups:", cross_split_exact_group_count)
print(" - near-duplicate pairs >= 0.95:", len(near_pairs_df))
print(" - cross-split near-duplicate pairs >= 0.95:", int(near_pairs_df["cross_split"].sum()))
